# Graframe playground

A scaffold for exploring an RDF graph with the **facet-based** query interface (`aq.graph()`).
This copy is pre-filled to run against **`DPR-full.ttl`** (a 223 + NAWI water model), but every
`# <-- EDIT ME` knob can be swapped for your own graph/ontology.

The whole surface is two symmetric moves plus introspection:

| move | what it does | rows |
|------|--------------|------|
| `.facets()` | show the neighbourhood: what predicates/objects are reachable next | (read-only) |
| `.having(step, **φ)` | **stay** — keep current nodes that have such an edge (existential) | never multiplies |
| `.follow(step, **φ)` | **move** the cursor to the neighbours along an edge | adds a column |

Everything compiles to SPARQL — call `.to_sparql()` on any selection to see exactly what runs.

## 1. Connect

In [1]:
import polars as pl
from acquirium.Client.acquirium import Acquirium
from acquirium.Graframe import P, Reasoning  # P = property-path helper for virtual edges

pl.Config.set_fmt_str_lengths(70)   # so long URIs aren't truncated to look identical

# Point this at your running Acquirium server (see README / `make up`).
aq = Acquirium(server_url="localhost", server_port=8000)

g = aq.graph()            # Graframe root, default reasoning (transitive subclass)
g

## 2. Load your graph

`insert_graph` accepts a file path **or** raw turtle text (read client-side, so the path is
relative to this notebook). `replace=True` swaps out the current main graph.

In [2]:
GRAPH_PATH = "../DPR-full.ttl"   # <-- EDIT ME

aq.insert_graph(GRAPH_PATH, format="turtle", replace=True)
print("graph version:", aq.graph_version())

graph version: 10


### Prefixes
Graframe resolves CURIEs (`prefix:Local`) against the server's bound namespaces. These are set
authoritatively in `acquirium.toml` under `[prefixes]` (so `s223:` etc. are stable regardless of
what the loaded RDF declared).

> **Note:** the client caches `/namespace/list`. If you changed `[prefixes]` and rebuilt the
> server, **restart this kernel** so a fresh client picks up the new names.

In [3]:
import requests
ns = requests.get(f"{aq.client.base_url}/namespace/list").json()
{k: ns[k] for k in ["s223", "nawi", "qudt", "qk", "unit", "brick"] if k in ns}

{'s223': 'http://data.ashrae.org/standard223#',
 'nawi': 'urn:nawi-water-ontology#',
 'qudt': 'http://qudt.org/schema/qudt/',
 'qk': 'http://qudt.org/vocab/quantitykind/',
 'unit': 'http://qudt.org/vocab/unit/',
 'brick': 'https://brickschema.org/schema/Brick#'}

## 3. Seed a selection

A **Selection** is a set of focus nodes (a bindings table with a cursor). Seed it from a class,
specific node URIs, or everything.

In [4]:
SEED_CLASS = "s223:Sensor"   # <-- EDIT ME  (try nawi:Pump, nawi:Filter, nawi:Tank, s223:Junction)

sel = g.instances(SEED_CLASS)     # instances of the class (+ subclasses, by default)
# sel = g.nodes("wbs:some-node-1", "wbs:some-node-2")   # or specific nodes
# sel = g.everything()                                    # or every subject in the graph

print(sel)
print("count:", sel.count())

<Selection focus=n0 marks=[-] patterns=1>
count: 7


## 4. Facets — what can I query next?

Facets summarise the neighbourhood of the current nodes. `support` = how many of your current
nodes can take that step; `edges` = total matching edges. Direction defaults to **both**.

In [8]:
sel.facets().show()                       # group by predicate (in + out)
# sel.facets(by="pred-obj").show()        # group by (predicate, object value)
# sel.facets(by="pred-obj-type").show()   # group by (predicate, object's rdf:type)
# sel.facets(direction="out", limit=15).show()

                 Facets (by=predicate)                 
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ dir ┃ predicate                   ┃ support ┃ edges ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ out │ s223:hasObservationLocation │       7 │     7 │
│ out │ rdf:type                    │       7 │     7 │
│ out │ s223:observes               │       7 │     7 │
└─────┴─────────────────────────────┴─────────┴───────┘

<Facets by=predicate rows=3>

In [9]:
# Facets as a DataFrame, or just the predicate list, if you'd rather work programmatically:
sel.facets(by="pred-obj-type").to_polars()
# sel.facets().predicates()

direction,predicate,object_type,support,edges
str,str,str,i64,i64
"""out""","""s223:hasObservationLocation""","""s223:Connection""",7,7
"""out""","""rdf:type""","""sh:NodeShape""",7,7
"""out""","""s223:observes""","""s223:QuantifiableObservableProperty""",7,7
"""out""","""rdf:type""","""s223:Class""",7,7
"""out""","""s223:hasObservationLocation""","""s223:Pipe""",7,7


## 5. Refine (stay) vs pivot (move)

From the facets above, sensors have `s223:observes` (→ a property) and
`s223:hasObservationLocation`. `refine` narrows the *current* nodes; `pivot` walks to the neighbours.

In [10]:
STEP = "s223:observes"   # <-- EDIT ME to a predicate from the facets

# Narrow: keep only sensors that HAVE such an edge
print("have", STEP, ":", sel.having(STEP).count())

# Move: hop to the objects on the other end, then explore from there
props = sel.follow(STEP)
print("reached", props.count(), "property nodes")
props.facets(direction="out").show()

have s223:observes : 7
reached 7 property nodes


             Facets (by=predicate)              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ dir ┃ predicate            ┃ support ┃ edges ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ out │ qudt:hasUnit         │       7 │     7 │
│ out │ rdf:type             │       7 │     7 │
│ out │ qudt:hasQuantityKind │       7 │     7 │
│ out │ rdfs:comment         │       7 │     7 │
│ out │ s223:ofSubstance     │       5 │     5 │
│ out │ s223:ofMedium        │       1 │     1 │
└─────┴──────────────────────┴─────────┴───────┘

<Facets by=predicate rows=6>

Object filters (φ) work on both `refine` and `pivot`:

- `value=` / `in_=[...]` — object equals a URI/CURIE/literal (or one of a list)
- `is_a=` — object has an rdf:type (or one of a list)
- `min=`, `max=` — numeric range on a literal object
- `matching=other_selection` — object must be a member of another selection (join)
- `direction="in"` — follow the edge backwards (inverse). `"^pred"` works too.
- pass a **list** of predicates for alternation, e.g. `pivot(["s223:connectedTo", "s223:cnx"])`

In [11]:
# e.g. observed properties that measure a Concentration:
conc = sel.follow("s223:observes").having("qudt:hasQuantityKind", value="qk:Concentration")
print("concentration properties:", conc.count())

concentration properties: 5


In [16]:
conc.facets()

<Facets by=predicate rows=8>

## 6. Correlated & disjunctive constraints

`where(...)` holds the focus fixed while checking a branch (never multiplies rows).
`any_of(...)` is disjunction. `without(...)` is negation.

In [ ]:
# Sensors that observe a Concentration property (the sensor stays the focus):
result = g.instances("s223:Sensor").where(
    lambda s: s.follow("s223:observes").having("qudt:hasQuantityKind", value="qk:Concentration")
)
print("count:", result.count())
print(result.to_sparql())

In [ ]:
# Sensors observing a Concentration OR a Turbidity property:
either = g.instances("s223:Sensor").any_of(
    lambda s: s.follow("s223:observes").having("qudt:hasQuantityKind", value="qk:Concentration"),
    lambda s: s.follow("s223:observes").having("qudt:hasQuantityKind", value="qk:Turbidity"),
)
either.count()

## 7. Waypoints — build a result table

`mark(name)` labels the current column; `to(name)` jumps the cursor back to it;
`select(cols...)` projects the marked columns into a DataFrame (this is where a join is intended).

In [ ]:
table = (
    g.instances("s223:Sensor").mark("sensor")
     .follow("s223:observes").mark("property")
     .follow("qudt:hasQuantityKind").mark("quantity")
     .to("sensor")
     .follow("s223:hasObservationLocation").mark("location")
)
table.select("sensor", "property", "quantity", "location")

## 8. Property paths (virtual edges)

Compose multi-hop / transitive edges with `P(...)` and combinators
(`.then`, `.or_`, `.plus`, `.star`, `.opt`, `.inverse`). Pass the **full URI** to `P`
(combinators don't call the server); use `aq.client.expand_uri("curie")` if you have a CURIE.

In [ ]:
connected_to = aq.client.expand_uri("s223:connectedTo")
downstream = P(connected_to).plus()          # connectedTo, one-or-more hops

reach = g.instances("nawi:Pump").follow(downstream)
print("reachable from pumps via connectedTo+:", reach.count())
reach.facets(by="pred-obj-type", direction="out").show(8)
# print(reach.to_sparql())

## 9. Terminals & reasoning

Pull results out, or inspect the query.

In [ ]:
sel.nodes()[:10]      # focus node URIs (list of str)
# sel.count()         # how many
# sel.frame()         # single-column polars DataFrame (CURIE-compacted)
# print(sel.to_sparql())

In [ ]:
# Turn off subclass reasoning if you want exact-type matching only:
g_exact = aq.graph(reasoning=Reasoning(subclass=False))
print("with subclass reasoning:   ", g.instances(SEED_CLASS).count())
print("exact type only:           ", g_exact.instances(SEED_CLASS).count())